# Module 1: Agentic RAG

Lessons from [`course/01-agentic-rag`](../../course/01-agentic-rag/) in this notebook, one section per lesson:

- **Lesson 01** - Introduction (`01-intro.md`)
- **Lesson 02** - Environment (`02-environment.md`)
- **Lesson 03** - RAG (`03-rag.md`)
- **Lesson 04** - The Course FAQ Dataset (`04-dataset.md`)
- **Lesson 05** - Search (`05-search.md`)
- **Lesson 06** - Building the Prompt (`06-building-prompt.md`)
- **Lesson 07** - The LLM (`07-llm.md`)
- **Lesson 08** - RAG Helper (`08-rag-helper.md`)
- **Lesson 09** - Data Ingestion (`09-data-ingestion.md`)
- **Lesson 10** - Wrap-up of Part 1 (`10-rag-next-steps.md`)
- **Lesson 11** - Agents (`11-agents-intro.md`)
- **Lesson 12** - Quick RAG Revision (Optional) (`12-rag-revision.md`)
- **Lesson 13** - Function Calling (`13-function-calling.md`)
- **Lesson 14** - The Agentic Loop (`14-agentic-loop.md`)
- **Lesson 15** - ToyAIKit (`15-frameworks.md`)
- **Lesson 16** - Other Frameworks (`16-other-frameworks.md`)

---

# Lesson 01: Introduction

Source: [`01-intro.md`](../../course/01-agentic-rag/01-intro.md) · Video: [YouTube](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Before you start, it's important to go through the course documentation:

- [Zoomcamp Logistics](https://datatalks.club/docs/courses/zoomcamp-logistics/)
- [LLM Zoomcamp](https://datatalks.club/docs/courses/llm-zoomcamp/)

Places where you can find me:

- [My substack](https://aishippingblog.com/)
- [LinkedIn](https://www.linkedin.com/in/agrigorev/)
- [X](https://x.com/Al_Grigor)

## LLMs

An LLM (Large Language Model) is a neural network trained on massive
amounts of text. Given a prompt, it generates a continuation - a
plausible next piece of text.


Think of your phone. When you type "how are" in WhatsApp, it suggests
"you" as the next word. "How are you" is the most common continuation.
Your phone uses a simple language model for that. It predicts the next
word based on what you typed so far.

A large language model does the same thing, but at a much larger scale.
It has billions of parameters and is trained on most of the text on the
internet. When it predicts the next word, it feels like you're talking
to an intelligent being. It understands what you ask and gives
meaningful answers.

In this course, we treat LLMs as black boxes. We won't look inside or
cover the theory, and we won't host a model ourselves. We use an LLM
provider and call it over an API. For us, an LLM is a box: text goes in,
text comes out.

But LLMs have limitations:

- Knowledge cutoff: they only know what was in their training data.
  If you ask about something that happened after training, they won't
  know - or worse, they'll make something up.
- No access to your data: they can't see your documents, databases,
  or internal systems unless you provide that information.
- Hallucinations: they sometimes produce confident-sounding answers
  that are simply wrong.

## The project

RAG solves these problems by giving the LLM relevant documents at
question time. We don't hope the model memorized the answer. We
retrieve the right information and hand it to the LLM, and the model
generates a grounded response. This lets us inject knowledge the model
never saw during training. That's why RAG is still the most common way
people use LLMs in the industry.

To make this concrete, we build a FAQ agent for our course. A student
asks something like "when does the course start?" and the agent answers
from the FAQ data we prepared.

![Overview of the course RAG project: a question is searched against FAQ documents, passed as context to an LLM, and returned as an answer](../../course/01-agentic-rag/images/01-intro-01-rag-project-overview-imagegen.png)

This module has two parts.

In Part 1 (the next 9 lessons) we will:

- Understand what RAG is and how it works
- Build a search engine over a real FAQ dataset
- Write a prompt that combines the user's question with search results
- Wire it all together into a working RAG pipeline
- Split ingestion and query into separate processes


In Part 2, we make the pipeline agentic. The LLM decides when and
what to search, instead of running the same fixed flow every time.

The final code from this module is available in the
[code/](../../course/01-agentic-rag/code/) directory.

---

# Lesson 02: Environment

Source: [`02-environment.md`](../../course/01-agentic-rag/02-environment.md) · Video: [YouTube](https://www.youtube.com/watch?v=3U4gBrmkZyM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

For this module, all you need is Python with Jupyter.

## Prerequisites

You need the following:

- Python (3.14 or later)
- An [OpenAI account](https://openai.com/) (or an OpenAI-compatible
  provider like Groq, Gemini, or Ollama)
- Basic familiarity with Python and the command line

## Creating the project

We'll start from scratch with no cloning needed - you'll create the
project yourself, step by step, either locally or on GitHub Codespaces.

## Creating the project locally

First, install uv - it's a Python package manager, and I switched all my
projects to it because it's fast and convenient. Once I started using
it, I never wanted to go back.

On Mac or Linux:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

On Windows:

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

(You can also use `pip install uv` if you prefer.)

Create an empty folder for the project and initialize it:

```bash
mkdir llm-zoomcamp-2026-code
cd llm-zoomcamp-2026-code
uv init
```

This creates a `pyproject.toml` and a basic project structure.

## Creating the project on GitHub Codespaces

We suggest using Codespaces because everyone gets the same Ubuntu,
Python, and Docker. That makes it easier to help each other when
problems come up.

Setup:

- Create a new repo on GitHub. Name it whatever you want, for example
  `llm-zoomcamp-2026-code` or `introduction-to-rag`, and add a README.
- Open the repo, click the green `<> Code` button, switch to the
  Codespaces tab, and create a codespace.



You now have a remote environment running in Codespaces. By default it
opens an in-browser editor, but you can connect VS Code on your desktop
for a better experience. Click `Codespaces` in the bottom-left corner and
pick "Open in Visual Studio Code Desktop" from the dropdown.


Once VS Code opens, press `` ctrl+` `` to bring up the terminal and
initialize the project the same way as locally:

```bash
pip install uv
uv init
```

## Adding dependencies
Now add the dependencies we'll need:

```bash
uv add requests minsearch openai jupyter python-dotenv
```

This installs:

- `requests` - to fetch the FAQ dataset from the internet
- `minsearch` - a simple in-memory search engine for indexing and
  searching text
- `openai` - the OpenAI API client for calling the LLM
- `jupyter` - the notebook environment where we'll write and run code
- `python-dotenv` - to load API keys from a `.env` file

## Setting up API keys

We need an API key to talk to the LLM. If you're using OpenAI, you'll
need to deposit some money first. The minimum is $5 (as of June 2026).
This lesson costs well under 10 cents to run, so that $5 goes a long
way.

I also recommend creating a separate OpenAI project for the course.
Then you can open the usage page and see exactly how much you spent
here, apart from your other work.


The safest way to store the key is in a `.env` file that never gets
committed to git.

Create a `.env` file in your project folder and put your API key in
it:

```bash
OPENAI_API_KEY=sk-YOUR_KEY_HERE
```

Now add `.env` to `.gitignore` to make sure you never accidentally
commit your key:

```bash
.env
```


Never commit `.env` to git. Treat the API key like a password. If it
leaks, someone else can run up charges on your account.

## Starting Jupyter

Start Jupyter:

```bash
uv run jupyter notebook
```

Create a new notebook. Throughout the course, you'll copy code from
the section notes into notebook cells.

Check that the OpenAI client works:

In [2]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

If you see an error, make sure the key in your `.env` file is
correct.

For Groq or other OpenAI-compatible providers, add the key to
`.env`:

```bash
GROQ_API_KEY=your_key_here
```

And configure the client:

In [ ]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

## (Optional) Auto-loading .env with dirdotenv

If you don't want to call `load_dotenv()` in every notebook, use
[dirdotenv](https://github.com/alexeygrigorev/dirdotenv).

It loads `.env` files automatically when you `cd` into a directory:

```bash
uv tool install dirdotenv
echo 'eval "$(dirdotenv hook bash)"' >> ~/.bashrc
```

Restart your terminal, and now whenever you enter the project
directory, the variables from `.env` are loaded automatically. No
`load_dotenv()` needed.

---

# Lesson 03: RAG

Source: [`03-rag.md`](../../course/01-agentic-rag/03-rag.md) · Video: [YouTube](https://www.youtube.com/watch?v=JktYwBIDErk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

We run free Zoomcamp courses at DataTalks.Club on data engineering,
machine learning, MLOps, and other topics. Each course has its own
FAQ document with common questions and answers.

Some of these documents have over 300 questions. Students ask us
things in Slack like "Can I still join after the course started?" or
"How do I get a certificate?" Finding those answers in the FAQ is
tedious.

We want a bot that takes all this knowledge and answers student
questions in natural language.

In this module, we'll build that system. But first, let's see why we
can't send the question straight to an LLM and call it a day.

## Plain LLMs lack our data

First, let's define a function to talk to the LLM:

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-6-luna",
        input=prompt
    )
    return response.output_text

This is our black box - text goes in, text comes out.

Let's test it:

In [4]:
llm("Hey, what's up?")

"Hey! Not much—I'm here and ready to help. What’s up with you?"

It replies with something. The LLM works.


Ask it a course-specific
question:

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Possibly! It depends on the course’s enrollment dates and whether late registration is allowed. Check the course page for an **Enroll/Join** option, or tell me which course you mean and I can help you figure it out.


The LLM gives a generic answer. It might say "you can usually join" or
"check the course website." It doesn't know about our specific Zoomcamp
courses, their enrollment policies, or their schedules. It tries to be
helpful, but has no idea about actual enrollment status or policies.


This is different from a question like "how do I cook salmon?" - the
LLM knows the answer because cooking salmon is common knowledge. But
our courses are not in the training data.

## Adding context manually

More context can fix this. The FAQ website has questions and answers
about our courses.


Copy some of that content into the prompt:

In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

Notice the prompt doesn't end with `Answer:`. With older models like
GPT-3 we added that to nudge the model into completing the sentence.
Modern models don't need the hint, so we drop it.

Build a prompt that includes both the question and the context:

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

Instead of sending the raw question to the LLM, we send this prompt:

In [8]:
answer = llm(prompt)
print(answer)

Yes, you can join now. If you want a certificate, submit your project while submissions are still open.


After that, the answer is correct: "Yes, you can still join. If you want to
receive a certificate, you need to submit your project while
submissions are still open."


This is the answer we actually want to give to our students. What we
just did is nothing but RAG.

## Retrieval plus generation

RAG stands for Retrieval-Augmented Generation. Generation is the LLM
producing text, and retrieval is search. We retrieve relevant documents
from our knowledge base and use them to augment what the LLM generates.
That search step is what gives the LLM the context it needs to answer
correctly.

![Clean RAG architecture: a user sends a question to an assistant, which consults a knowledge base and returns an answer](../../course/01-agentic-rag/images/03-rag-08-rag-architecture-sketch-imagegen.png)

What we just did was naive. I knew in advance which FAQ entry held the
answer and pasted it in by hand. What we want instead is to perform
search automatically. We take the student's question, find the most
relevant documents, and send those to the LLM.

In code, it looks like this:

In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

That's the entire architecture. It comes down to three components.

The pieces are search, the prompt, and the LLM:

- search
- prompt
- LLM


```mermaid
flowchart TD
    U([User])

    APP[Application]

    DB[(DB)]
    DOCS[[D1 ... D5]]

    PROMPT[Build Prompt<br/>Question + Context]

    LLM[LLM]

    ANSWER([Answer])

    U -->|Question| APP

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
    DOCS --> APP

    APP --> PROMPT
    PROMPT --> LLM

    LLM --> ANSWER
    ANSWER --> U
```

The LLM only sees the documents we hand it, so its answers are grounded
in our data. If the right document is retrieved, the answer is good. If
it's not, the LLM gets the wrong context and the answer is wrong. Your
model is only as good as your retrieval, so search quality matters a
lot for RAG.

The database and the LLM can be anything. In this course we use
minsearch and then sqlitesearch for search, and OpenAI for the LLM. But
you can swap any component for another and see what works better.

Because each piece is independent, RAG stays flexible. To use Anthropic
instead of OpenAI, you swap the LLM call. To use Elasticsearch instead
of minsearch, you swap the search call. Nothing else changes.

In the next section, we'll look at the dataset we'll use for our FAQ
knowledge base.

---

# Lesson 04: The Course FAQ Dataset

Source: [`04-dataset.md`](../../course/01-agentic-rag/04-dataset.md) · Video: [YouTube](https://www.youtube.com/watch?v=Mx6EqvzVDz0&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

Before we build the RAG pipeline, let's get familiar with the data
we'll use as our knowledge base.

We run these courses every year, and students keep asking the same
questions in Slack. We collected those into an FAQ so people can find
answers before asking. Some courses have run for five cohorts, so the
FAQ grows large and searching it by hand gets tedious. That's exactly
the problem our RAG system will solve.


The FAQ data is available as JSON from the DataTalks.Club website. I
maintain that site, so I made the data available at a JSON endpoint we
can fetch directly.


Let's fetch it:

In [9]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

This returns a list of courses. Each course has a `path` field that
points to its FAQ data.

Let's fetch all the FAQ documents from all
courses:

In [10]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1456

Each entry has:

- `id` - unique identifier
- `course` - course slug (e.g., `machine-learning-zoomcamp`)
- `section` - which section of the course
- `question` - the FAQ question
- `answer` - the FAQ answer


Let's look at one:

In [11]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

You should see something like:

In [ ]:
{
    "id": "0e38656cfb",
    "course": "machine-learning-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "How do I submit homework?",
    "answer": "- Do the tasks locally\n- Publish your code ..."
}

Each course has a slug - a short identifier used in URLs. For example,
`machine-learning-zoomcamp`, `data-engineering-zoomcamp`, etc. We'll
use these slugs for filtering in search.

## Using this data

In the RAG pipeline, this dataset is our knowledge base:

1. We index all the documents (the search step)
2. When a student asks a question, we search the index
3. The search returns the most relevant FAQ entries
4. We give those entries to the LLM as context
5. The LLM generates an answer based on the context

The `question` and `answer` fields contain the text we'll search
through. The `course` field lets us filter by course. For example, if a
student asks about the data engineering course, we skip results from
the ML course. The `section` field helps with ranking - knowing which
part of the course a question belongs to is useful context.

## A note on data preparation

In our case, the data is already prepared. I maintain this FAQ website
and made sure the data comes back in a convenient JSON format. So we
don't need to do much to get it ready. By the way, I cleaned a lot of
this data with the help of an LLM. That's a handy use case on its own.

Don't let that fool you. In reality, data preparation is often the most
time-consuming part of building a RAG system. You may need to scrape
websites, parse PDFs, and clean and chunk documents. That work isn't
visible here, but I did plenty of it ahead of time.

We keep the focus on the GenAI side in this course. In your own
projects, expect to spend real time on data preparation before you get
to this point.

In the next section, we'll build the search index.

---

# Lesson 05: Search

Source: [`05-search.md`](../../course/01-agentic-rag/05-search.md) · Video: [YouTube](https://www.youtube.com/watch?v=GYgpNKiuCJU&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

## Search basics

At its core, every search engine does the same thing. It takes a query,
scores every document for similarity, and returns the top results.

Formally, there is a similarity function:

```python
score = sim(query, document)
```

For each document in the database, you compute this score. Then you
rank all documents by score and return the top N. What makes a search
engine different from another search engine is what `sim` actually
computes.

- text/lexical search (covered in this section): `sim` counts how
  many words the query and the document share. It looks at the surface
  form, the actual words, and matches them exactly.

- vector/semantic search (covered in [module 2](../../course/02-vector-search/)):
  `sim` compares the meaning of the query and the document. Same
  function, different similarity measure.

Consider these two questions:

- "Can I still join the course after the start date?"
- "Is it possible to enroll late?"

They mean the same thing, but share almost no keywords. "Join" is not
"enroll", "course" is absent, "start date" is not "late". A text search
engine would struggle to match them, because it only sees words.

We'll see how vector search solves this later. For now, let's build text
search with minsearch.

## Indexing with minsearch

We already have the `documents` list from the previous section. Now
let's index it.


Searching matters because we have around 1100 documents. Sending all
of them to the LLM would be expensive and slow. The model would get
confused with too much data. Search finds the most relevant documents
to send instead.

There are many search libraries you can use - Apache Lucene,
Elasticsearch, Solr, and others. But these are somewhat heavy. For
example, to run Elasticsearch, you need to start a Docker container.


[minsearch](https://github.com/alexeygrigorev/minsearch) is a simple
in-memory search engine. It's lightweight, so it runs anywhere Python
runs, including Google Colab where you can't start a Docker container.
It's a toy implementation, not production ready, but it illustrates how
search engines work and it gives good results.

I should add a disclaimer: I wrote this library and I maintain it. It
started as a single Python file in the first edition of LLM Zoomcamp. I
wanted to show that keyword search isn't magic. We wrote it together as
part of the [Build a Search Engine](https://www.youtube.com/watch?v=nMrGK5QgPVE) workshop
(see the [code](https://github.com/alexeygrigorev/build-your-own-search-engine)).


It turned out useful beyond teaching. After two years across many
projects, it's pretty reliable for small datasets.

The concepts in minsearch are the same as in Elasticsearch (which
comes from Lucene): text fields, keyword fields, boosting, filtering. I
borrowed those terms from Elasticsearch on purpose, since I wanted a
lightweight stand-in for it. So what you learn here transfers directly.

We'll index the `question`, `section`, and `answer` fields as text
(they'll be tokenized and ranked), and the `course` field as a
keyword (for filtering).

The index tokenizes text fields and treats keyword fields as exact strings.

Text fields are the fields you search through. When you type a query,
the search engine looks at these fields and tokenizes them. It splits
text into words, lowercases them, removes stop words, and ranks the
results by relevance. So `question`, `section`, and `answer` are text
fields.

Keyword fields are for exact matching. Think of a SQL query like
`SELECT * FROM index WHERE course = 'data-engineering-zoomcamp'`. The
results must come from the specified course, no matter what ranking or
boosting you do for text fields.

You use keyword fields to restrict the search space to a particular
subset. In our case, we have four courses. Say you're taking the LLM
course and ask a question. You don't want answers from the MLOps or
machine learning courses mixed in.

In [12]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

That's it, the index is built. The `fit` name comes from scikit-learn,
where you fit a model on data. Here we fit an index on our documents.

## Trying a search

Let's try a search with the question we used before:

In [13]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

We get back 5 results from the LLM Zoomcamp FAQ. The best match is the FAQ
entry "I just discovered the course. Can I still join?" which is exactly what
we need.

Here are all the questions:

In [14]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Does the course certificate show the number of course hours?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?']

We see questions about joining the course, registration, certificates,
and more. These are the candidate documents we'll send to the LLM.

We used `boost_dict` to give the `question` field more weight (2.0
instead of the default 1.0) and `section` less weight (0.5). Query
words appearing in the question field is a stronger signal than them
appearing in the section name.

We used `filter_dict` to only return results from the LLM Zoomcamp
course. Without this filter, we'd get results from all four courses.

## Boosting fields

Not all fields are equally important. The `question` field is usually
more relevant than `section` for matching. Query words appearing in the
question is a stronger signal than them appearing in the section name.

minsearch supports field boosting to reflect this:

In [15]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

All fields have a default boost of 1. Giving `question` a boost of 2
means it counts two times as much. Take a question about certificates.
The word "certificate" in the question field now weighs twice what it
does in the answer.


Giving `section` 0.5 means it counts half as much, since a match there
tells us less. This is the same boosting mechanism used by
Elasticsearch and Lucene.

Try different boost values and see how the results change.

## Filtering by course

Sometimes you want to restrict the search to a specific course.

minsearch supports keyword filtering:

In [16]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

This only returns documents from the MLOps Zoomcamp. Try a few
different queries and courses to get a feel for the results.

In [17]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Course: How do I start?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?']

In [18]:
question

'I just discovered the course. Can I join now?'

## Wrapping it in a function

Let's wrap the search in a `search` function - the first component of
our RAG pipeline:

In [19]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

By default it searches the LLM Zoomcamp FAQ.


You can pass a
different course slug to search other courses:

In [20]:
search_results = search(question)

In [21]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

---

# Lesson 06: Building the Prompt

Source: [`06-building-prompt.md`](../../course/01-agentic-rag/06-building-prompt.md) · Video: [YouTube](https://www.youtube.com/watch?v=DV4e2n-dIv0&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

The LLM doesn't see our documents unless we pass them in. So we need
to build a prompt that includes the user's question and the search
results.

When we build AI systems, we usually split the prompt into two parts:

- Instructions (also called the system prompt): this tells the LLM how
  to behave. It never changes, so it's the same for every request.
- User prompt: this changes with every request. It carries the actual
  question and the retrieved context.

We split them because the instructions are fixed and the user prompt is
not. Keeping them apart makes the fixed part easy to reuse and the
changing part easy to build fresh each time.

## Instructions

The instructions tell the LLM its role and how to answer:

In [22]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

This is what grounds the answer in our data and reduces hallucinations.

## The user prompt template

The user prompt template has placeholders for the question and the
context:

In [23]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

## Building the context

The `context` is a formatted string with all the search results:

In [24]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

Each document becomes a block with the section, question, and answer.
This format makes it easy for the LLM to read. We turned a list of
dictionaries into one string. It's a small preprocessing step before we
send the data to the LLM.

## Building the prompt

Now we combine the question with the context into the user prompt:

In [25]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

Let's try it:

In [26]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI w

You should see a prompt with the question at the top and several
FAQ entries below it. This is exactly what we'll send to the LLM.

The prompt looks something like:

```text
Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework...

...
```


The prompt is the bridge between search and the LLM. A bad prompt lets
the LLM ignore the context and hallucinate. A good prompt keeps the
answer grounded.

Prompt engineering is part art, part science. You experiment, try
different things, and see what works. Later in the course we cover
evaluation metrics so you can measure how well your prompt performs
instead of guessing. For now, this template is a good starting point.

---

# Lesson 07: The LLM

Source: [`07-llm.md`](../../course/01-agentic-rag/07-llm.md) · Video: [YouTube](https://www.youtube.com/watch?v=KHePGkeFn54&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

The last component of our RAG pipeline is the LLM. It takes
the prompt we built and generates an answer.

## Sending the prompt to the LLM

We have the prompt from the previous section.

We send it to the
LLM:

In [27]:
response = openai_client.responses.create(
    model="gpt-6-luna",
    input=prompt
)

In [31]:
response

Response(id='resp_046aabe5e7edfb07006ae0dd39891c87d2a7a39e7d55636b0f', access_programs=None, created_at=1793121593.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-6-luna', object='response', output=[ResponseReasoningItem(id='rs_046aabe5e7edfb07006ae0dd3a6e6087d2962ab395b2b495da', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABq4N07jAZravpO6jIifFOePs8pQwt5PzE11ctE8gkZJ35A2OQ-oazQTJ1Z7wn3c1H7pO6b3VwiC2QmLcDryGYoubDAkTDyOdoqhTMHfnMUCqTAVQ2M9QDmB6V9jaZFN6rXNMe5QFJ51fgki5ZimJOzjiIomuUeu9qbR_HrT7nIhT4AxfrQSXxo5WNmO0kP9R88UhIWzgjaq8hJpVGGBR_xjtXT3v44RuwegvAXLaSRD28iA6JLUgdrMTHtgvlzVSogph09sncGXzULYBfx_jvuhjg0NUAZ9hD7flXDH-wLBDibpypuvqJrf1h7ImlkzdmfvoGpTwWJoSC_PYlQPi3W7Q8TRCwXacv0FVVFQPD3gi70zfvxC0HAPIyXGb2csSl5t0Oit2Cao7b8LVW6emhiceYZz-HJDo1szLBqnOHTL6jvYcMC4SB-En3rfzVkH6DARP1aO3w9tS8szMFDEUvexJosSG1qpg-AD85r0Oyqf_kFBYqZUcOTk_2U0JzsvDURJjIqLPOrSkpZZ1sfhrAyJXlq5v5Cal5kE54QCvMyWeNwjg5Op5j4vIEPlyB6Ue3w8ZMmGpFb0UH45cwjEaAjDoGZKDJEN9fRl7Epzoh

We use OpenAI's Responses API (`openai_client.responses.create`). OpenAI
has two APIs: chat completions and responses. Chat completions is the
older one, and it's now considered legacy. When the first edition of
this course started, the Responses API didn't exist, so we used chat
completions. Now we prefer responses because it's more convenient.

There's a catch worth knowing. Many other providers like Groq and
Gemini give you an OpenAI-compatible client. But they expose chat
completions, not responses. So if you switch providers, you keep the
OpenAI client but call `chat.completions` instead of `responses`.

## Exploring the response

The response is a Pydantic object. The answer is in `response.output` -
a list of output items.

The first one is the message:

In [32]:
response.output[1]

ResponseOutputMessage(id='msg_046aabe5e7edfb07006ae0dd3b1a7087d2a41a6f117d739fd0', content=[ResponseOutputText(annotations=[], text='Yes—you can join and start learning now; registration isn’t required. To earn a certificate, you’ll need to submit your capstone project and complete the required peer reviews while a live cohort is accepting submissions.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

The message has a `content` list, and the text is in the first item:

In [33]:
response.output[1].content[0].text

'Yes—you can join and start learning now; registration isn’t required. To earn a certificate, you’ll need to submit your capstone project and complete the required peer reviews while a live cohort is accepting submissions.'

That's quite a journey to reach one string.

The shortcut spares us all of it:

In [34]:
response.output_text

'Yes—you can join and start learning now; registration isn’t required. To earn a certificate, you’ll need to submit your capstone project and complete the required peer reviews while a live cohort is accepting submissions.'

Same result, less code. The answer should be something like: "Yes, you
can still join. If you want to receive a certificate, make sure to
submit your project while submissions are still open."

The usage counts tell you how many tokens the request consumed:

In [35]:
response.usage

ResponseUsage(input_tokens=506, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=109, output_tokens_details=OutputTokensDetails(reasoning_tokens=61), total_tokens=615)

You'll see something like:

```text
ResponseUsage(input_tokens=334, output_tokens=39, total_tokens=373)
```

## Calculating the price

You can use different models.

In this course we'll use
[gpt-5.4-mini](https://developers.openai.com/api/docs/models/gpt-5.4-mini):

- Input: $0.75 per million tokens
- Output: $4.50 per million tokens


Let's calculate the cost of the request we just made:

In [36]:
input_price = 0.1 / 1_000_000
output_price = 0.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00010510000000000001

This particular request costs a fraction of a cent. Even a full RAG
query with a long prompt stays under $0.01. We need to send a lot of
queries to even spend one cent. These models are cheap to play with.

The usage object also reports cached input tokens. Those are billed at
a lower rate when the same prompt prefix repeats.

## Message history

Previously we sent only one string as input and got back a response.
In practice, you typically send a message history - a list of messages
where each message has a role.

Think of a ChatGPT conversation. It starts with a hidden system prompt
that tells the LLM how to behave, one you never see. After that, your
messages and the LLM's replies alternate. The LLM has no memory of its
own, so it needs the full history passed in to continue the
conversation.

We won't build a multi-turn chat here. But we still use this message
format to separate our instructions from the user prompt.

We send two messages:

- `developer` - system-level instructions (how the LLM should behave)
- `user` - the actual prompt with the question and context

In [37]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-6-luna",
    input=message_history
)

This separates the fixed instructions from the user prompt, which
changes every request.

OpenAI accepts both `developer` and `system` for the instruction role.
There may be some difference between them, but in practice I don't see
it change the result either way. We use `developer` in this course.

## The LLM function

We can now put this together into an updated `llm` function.

It now
takes both instructions and the user prompt:

In [38]:
def llm(instructions, user_prompt, model="gpt-6-luna"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [39]:
prompt

'Question:\nI just discovered the course. Can I join now?\n\nContext:\nGeneral Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ: Does the course certificate show the number of course hours?\nA: No. The certificate does not state a total number of hours.\n\nModule 3: Orchestration\nQ: Why do we need orchestration / Kestra — can\'t I just run the code in a notebook?\nA: Notebooks are great for learning and experi

## Full RAG

With search, the prompt, and the LLM ready, we wire them together:

In [40]:
def rag(query, model="gpt-6-luna"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

The revised architecture:

```mermaid
flowchart TD
    U([User])

    APP[Application]

    DB[(DB)]
    DOCS[[D1 ... D5]]

    PROMPT[Build Prompt<br/>Question + Context]

    LLM[LLM]

    ANSWER([Answer])

    U -->|Question| APP

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
    DOCS --> APP

    APP --> PROMPT
    PROMPT --> LLM

    LLM --> ANSWER
    ANSWER --> U
```


Try it:

In [41]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes. You can join now. If you want a certificate, you’ll need to submit your capstone project and complete the required peer reviews while a live cohort is accepting submissions. Homework isn’t required for the certificate.


The answer should be based on the FAQ documents - not on the LLM's
general knowledge. The LLM read the search results and generated a
response grounded in our data.

## Try more questions

Try a few more:

In [42]:
rag("How do I get a certificate?")

'To get a certificate, you need to complete the course with a **live cohort**, pass the capstone project, and finish all peer reviews assigned to you by their deadline. Homework is optional.\n\nCertificates are generated manually in one batch after the cohort’s final project attempt and peer-review period are over—you don’t need to request one. Make sure your profile has the name you want printed. You’ll receive an email when the certificate is ready; check your spam folder if you don’t see it.'

Notice how the answers reference specific courses and sections. The
LLM reads from our knowledge base before answering - that's how RAG
works.

This approach is modular. You can swap the search backend, the prompt
template, or the LLM model. Nothing else needs to change. Later when
we replace minsearch with sqlitesearch, only the `search` function
changes.

---

# Lesson 08: RAG Helper

Source: [`08-rag-helper.md`](../../course/01-agentic-rag/08-rag-helper.md) · Video: [YouTube](https://www.youtube.com/watch?v=JxaC6Hrym6c&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lessons, we built the RAG flow piece by piece -
search, then the prompt, then the LLM call. The pipeline works, but
every time we want to use it, we need to repeat the same code.

We'll use this code throughout the course, so let's put it into two
reusable files:

- [ingest.py](../../course/01-agentic-rag/code/ingest.py) - loading data and building the search index
- [rag_helper.py](../../course/01-agentic-rag/code/rag_helper.py) - the RAG logic (search, prompt, LLM)

Then in notebooks, we just import from these files and use them.

![Reusable RAG helper architecture: ingestion builds an index, while RAGBase combines search, prompt building, and the LLM client to answer a question](../../course/01-agentic-rag/images/08-rag-helper-01-reusable-rag-helper-imagegen.png)

## ingest.py

This file handles data loading and index creation - everything we
need before we can search.

Create `ingest.py` with two functions:

In [ ]:
import requests
from minsearch import Index

def load_faq_data():
    docs_url = "https://datatalks.club/faq/json/courses.json"
    response = requests.get(docs_url)
    courses_raw = response.json()

    documents = []
    url_prefix = "https://datatalks.club/faq"

    for course in courses_raw:
        course_url = f"""{url_prefix}{course["path"]}"""
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()

        documents.extend(course_data)

    return documents

def build_index(documents):
    index = Index(
        text_fields=["question", "section", "answer"],
        keyword_fields=["course"]
    )
    index.fit(documents)
    return index

We'll use `load_faq_data()` to fetch the documents and `build_index()`
to create the minsearch index. Later, we'll add sqlitesearch support
to this same file.

## rag_helper.py

This file contains the RAG logic - the same functions we wrote in the
previous lessons, now organized as a class.

We use a class because `index` and `openai_client` are currently global
variables. Move the functions to a separate file and those globals
aren't there anymore. We could import them back, but that ties the file
to one specific index and one specific client. That makes the code hard
to reuse and adjust.

So we put the dependencies inside a class instead. The index and the
LLM client become constructor arguments. Now we can pass any index or
client we want when we create the object. And because it's a class, we
can subclass it later to override one piece without touching the rest.
For example, we can swap OpenAI for a local model.


Create `rag_helper.py`:

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

Now the class: `RAGBase`

In [ ]:
class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course="llm-zoomcamp",
        model="gpt-5.4-mini"
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model

The `index` parameter is anything with a `search` method, whether
minsearch, sqlitesearch, or something else. The other four parameters
all have defaults. You only pass `course`, `instructions`,
`prompt_template`, or `model` when you want to override the default
behavior. We swap the index later without touching any of the RAG code.

The `search` method delegates to the index:

In [ ]:
    def search(self, query, num_results=5):
        boost_dict = {"question": 3.0, "section": 0.5}
        filter_dict = {"course": self.course}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )

The `build_context` and `build_prompt` methods format the search
results:

In [ ]:
    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc["section"])
            lines.append("Q: " + doc["question"])
            lines.append("A: " + doc["answer"])
            lines.append("")

        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

The `llm` method sends the prompt to the LLM:

In [ ]:
    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text

And the `rag` method wires it all together:

In [ ]:
    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer

## Using it in a notebook

Now in a notebook, import from both files and put everything together:

In [1]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join and start learning now. If you want a certificate, you’ll need to submit your capstone project and complete the required peer reviews while a live cohort is accepting submissions.


We don't need to pass `instructions` - the default from `rag_helper.py` is
used.

You can override it if you want different behavior:

In [2]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

Try more questions:

In [6]:
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join after the course has started. To receive a certificate, you’ll need to submit your project while project submissions are still being accepted.'

In [5]:
assistant.rag("I just discovered the course. Can I join now?")

'Yes, you can still join. If you want a certificate, you’ll need to submit your project and complete the required peer reviews while a live cohort is accepting submissions.'

We'll use these two files throughout the course. In the next lesson,
we'll see how to add sqlitesearch support to `ingest.py` for a
persistent search index.

---

# Lesson 09: Data Ingestion

Source: [`09-data-ingestion.md`](../../course/01-agentic-rag/09-data-ingestion.md) · Video: [YouTube](https://www.youtube.com/watch?v=e0owGI2JV-s&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

So far, our RAG pipeline loads data and builds the search index at
startup. With minsearch, this is fine - our FAQ dataset is small, so
indexing takes less than a second. The entire pipeline runs in one
process.


This breaks down as the dataset grows. Fetching data takes time -
calling APIs, parsing files, cleaning text. With millions of
documents, the startup becomes slow. You don't want to wait minutes
every time your service restarts.

Minsearch is in-memory. It's a bunch of Python dictionaries bound to
the process where it's running. When you stop the process, the data
disappears, so you re-index every time you restart. That's wasteful if
indexing is slow or the data takes time to prepare.

So we separate ingestion from querying. One process writes the data to
a persistent search index. Another process reads from it. The two run
independently and only share the index between them.


The index survives restarts, so we ingest once and query as often as
we like. This is the ingestion step from data engineering. We move data
from its source into a target system the application can use.

You can use any persistent search backend for this, such as
Elasticsearch, OpenSearch, or Qdrant. In this module, we use
[sqlitesearch](https://github.com/alexeygrigorev/sqlitesearch), a
lightweight search library backed by SQLite FTS5. It has the same API
as minsearch, so it's a drop-in replacement that happens to be
persistent.


I picked SQLite because it asks nothing of you. It ships with Python,
so you don't add any dependency, and it has FTS5 (full text search)
built in. If you have Python, you already have a full text search
engine. Using FTS5 directly is a bit awkward, so I wrote sqlitesearch
as a convenient wrapper around it.

You can read more about the history of this library [here](https://aishippingblog.com/p/how-i-built-sqlitesearch-a-lightweight).

Install it:

```bash
uv add sqlitesearch
```

## Ingestion notebook

Create a new notebook called `sqlite-ingest.ipynb` (see [persistent_rag_ingest.ipynb](../../course/01-agentic-rag/code/persistent_rag_ingest.ipynb) for reference). This is the
ingestion process - it fetches data and writes it to a persistent
index.

First, load the data using the function from `ingest.py`:

In [ ]:
from ingest import load_faq_data

documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

Filter to just the LLM Zoomcamp documents:

In [ ]:
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

Now create a sqlitesearch index and add documents one by one with a
small delay (to simulate slow ingestion):

In [ ]:
import time
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

for doc in docs_llm:
    index.add(doc)
    print(f"""Added: {doc["question"][:60]}...""")
    time.sleep(0.5)

index.close()
print("Done. Index saved to faq.db")

Run this notebook. You'll see each document being added one by one.
When it's done, there's a `faq.db` file on disk with the entire index.
This file persists across restarts.

## Querying notebook

While the ingestion is running (or after it finishes), create another
notebook (see [persinsent_rag.ipynb](../../course/01-agentic-rag/code/persinsent_rag.ipynb) for reference).

Connect to the same database:

In [7]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

sqlite_index.count()

161

Check how many documents are in the index:

Run this cell a few times while the other notebook is still ingesting.
You'll see the number growing as ingestion progresses. This is normal
database behavior: one process writes, another reads, both at the same
time. With minsearch this is impossible, because the index lives in one
process's memory and nobody else can reach it.

Try a search:

In [8]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'The homework submission form is still open even though the deadline has passed — can I still submit?',
 'Can I submit homework after the deadline, or get a deadline extension?',
 'I missed the first homework - can I still get a certificate?']

## RAG with sqlitesearch

We use the `RAGBase` class from `rag_helper.py` with this sqlitesearch
index.

Because our RAG is modular, we just swap the search index - the
rest of the code stays the same:

In [10]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv

openai_client = OpenAI()
load_dotenv()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
)

This code skips both the `fit` call and the data loading. The index
is already populated by the ingestion notebook, so we just connect to
the database file.

Try it:

In [ ]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can join after the course has started. To receive a certificate, you’ll need to submit your capstone project and complete the required peer reviews while a live cohort is still accepting them.


The answer should be similar to what we got with minsearch. But now the
data comes from a persistent index - no fetching, no processing, no
indexing at startup. And we didn't have to rewrite any of the RAG
logic - just swapped the index.

The modular design splits the work cleanly:

- `ingest.py` handles data loading and indexing
- `rag_helper.py` handles the RAG pipeline
- the notebooks wire them together

This works because sqlitesearch follows the same API as minsearch -
both have a `search` method that takes a query, `boost_dict`,
`filter_dict`, and `num_results`. If the API were different, we'd
need to subclass `RAGBase` and override the `search` method to adapt
to the new backend.

## Comparing the two approaches

With minsearch (single process):

```text
Startup: fetch data -> parse -> index -> ready
Every restart: repeat all steps
```

With sqlitesearch (two processes):

```text
Ingestion (runs once): fetch data -> parse -> write to faq.db
Query (runs every time): open faq.db -> search -> ready
```

The full architecture:

```mermaid
flowchart TD

    subgraph ING["INGESTION"]
        direction LR
        FAQ[FAQ.json]
        INGESTOR[Ingestor<br/>parse, chunk, embed, metadata]
        FAQ --> INGESTOR
    end

    subgraph KB["KNOWLEDGE BASE"]
        DB[(DB)]
    end

    INGESTOR -->|Index Documents| DB
```

The ingestion process writes documents to the knowledge base.

![Clean architecture showing FAQ ingestion into a shared knowledge base and the RAG assistant querying it](../../course/01-agentic-rag/images/09-data-ingestion-06-annotated-architecture-sketch-imagegen.png)

The RAG assistant then reads from it:

```mermaid
flowchart TD

    subgraph RAG["RAG ASSISTANT"]
        U([🙂 User])
        APP[Application]
        DOCS[[D1 ... D5]]
        PROMPT[Build Prompt<br/>Question + Context]
        LLM[LLM]
        ANSWER([Answer])

        U -->|Question| APP
        DOCS --> APP
        APP --> PROMPT
        PROMPT --> LLM
        LLM --> ANSWER
        ANSWER --> U
    end

    subgraph KB["KNOWLEDGE BASE"]
        DB[(DB)]
    end

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
```

For our FAQ dataset, both produce good results. The difference
matters more at scale with diverse document lengths.

## Choosing an approach

Pick the right tool for your data:

- `minsearch`: single process, in-memory only, re-indexes on every
  startup. Use when data is small and indexing is fast.
- `sqlitesearch`: separate ingestion and query, file-based (SQLite),
  opens existing index. Use when data is large or ingestion is slow.

Use minsearch when you can load and index the data on startup without
noticeable delay. Switch to a persistent backend when ingestion takes
too long or when you need the index to survive restarts.

For larger production systems, use the same pattern with a different
backend:

- Elasticsearch
- OpenSearch
- Qdrant (vector database)
- Weaviate (vector database)

The architecture stays the same: one process ingests, another queries.

## Cleaning up

When you're done, close the database connection:

In [ ]:
sqlite_index.close()

Or just let Python clean it up when the notebook kernel shuts down.

---

# Lesson 10: Wrap-up of Part 1

Source: [`10-rag-next-steps.md`](../../course/01-agentic-rag/10-rag-next-steps.md) · Video: [YouTube](https://www.youtube.com/watch?v=RPXMz5p5fb8&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In Part 1 of this module, we:

- Learned what RAG is and why it matters: retrieve documents, build a
  prompt, let the LLM generate a grounded answer
- Built a search engine over a real FAQ dataset using minsearch
- Created a prompt template that combines the user's question with
  search results
- Wired search + prompt + LLM into a working RAG pipeline
- Split ingestion and query into separate processes with sqlitesearch

You now have a working RAG system and a clear mental model for how each
piece fits together. From here, the work is making each piece better.

![Roadmap from a working RAG pipeline to agents, vector search, Elasticsearch, or fine-tuning](../../course/01-agentic-rag/images/10-rag-next-steps-01-rag-roadmap-imagegen.png)

## Two directions forward

[Part 2 of this module: Agents](../../course/01-agentic-rag/11-agents-intro.md). Our pipeline runs
search once with the exact user query, unchanged. If that search
returns garbage, the LLM has no way to recover. An agent puts the LLM
in charge instead. It decides what to search for, how many searches to
run, and when to stop.


An agent also handles questions in another language. It translates the
query before searching, then translates the answer back afterward.

[Module 2: Vector Search](../../course/02-vector-search/). Keyword matches
are exact. Vector search matches by semantic meaning instead, which
helps when the user phrases things differently from the FAQ.

## Elasticsearch

Elasticsearch is the industry standard for text search.

It supports:

- Full-text search with BM25
- Filtering, aggregations, and faceting
- Vector search (dense and sparse)
- Distributed scaling
- Real-time indexing


It's heavier than sqlitesearch but handles production workloads at
scale. If you're building a real RAG system, Elasticsearch (or
OpenSearch) is a common choice for the search backend.


For an Elasticsearch tutorial, see the
[supplementary materials for Module 1](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2025/01-intro/elastic-search.md).

## Fine-tuning vs RAG

Instead of retrieving documents at query time, you could fine-tune
the LLM on your data.

Fine-tuning means taking a model's weights and adjusting them for
your specific use case.

This works, but it has downsides:

- It requires special hardware (GPUs) and tools we don't cover in
  this course
- It's difficult to update when new data arrives - you don't want to
  retrain the model every time a new FAQ entry is added
- The LLM already has internal knowledge, but RAG gives it access to
  information it wasn't trained on

RAG is more flexible, cheaper, and works with any LLM. In practice,
fine-tuning is rarely needed. I've never personally hit a case that
required it. When I analyzed around 2,500 job descriptions for my AI
engineering field guide, few asked for it. So focus on RAG first, and
reach for fine-tuning only when you genuinely need it.

## Moving forward

Try these next steps:

- Try different prompts and see how the answers change
- Add more data sources to the knowledge base
- Experiment with different LLM models (GPT-4o, Claude, Gemini, local
  models via Ollama)
- Try Elasticsearch as a search backend

---

# Lesson 11: Agents

Source: [`11-agents-intro.md`](../../course/01-agentic-rag/11-agents-intro.md) · Video: [YouTube](https://www.youtube.com/watch?v=6uG4_Ivv60E&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In Part 1 of this module we built RAG pipelines.

Every pipeline we wrote followed the same flow:

- search the FAQ,
- build a prompt with the results,
- send it to the LLM.

This returns good answers when the user's query matches the documents.
The search finds the right entry, the LLM reads it, and you get a
helpful reply.


Often, though, the search returns nothing useful.

- Maybe the user made a typo.
- Maybe they asked the question in an unusual way.
- Maybe they need information from two different searches.

We use lexical search here, so the search looks for an exact match.
One typo and it misses the entry it needed. In our pipeline there's
no recovery. The search runs once, and if it returns garbage the LLM
gets garbage. Our pipeline always does the same thing, no matter what.


Instead of routing the user question straight to search, we can hand
control to the LLM and let it drive.

The LLM is in charge now, and it can:

- fix typos
- search again with different terms
- ask the user a clarifying question

A fixed flow can't do any of this. Once we put the LLM in control,
our system becomes agentic, so it's flexible rather than rigid.


An agent uses an LLM to decide which actions to take and in which
order. Instead of a fixed flow, the LLM chooses what to do at each
step.

![Clean agentic retry flow: the LLM notices an empty search, retries with the corrected term, and answers](../../course/01-agentic-rag/images/11-agents-intro-04-agentic-flow-diagram-imagegen.png)

In Part 2 of this module, we'll cover:

- Function calling, so we can give the LLM tools it can use
- The agentic loop, where the LLM decides when to call a tool, when
  to call another one, and when to stop and answer
- Frameworks, the libraries that run this loop for you

We build on top of the RAG pipeline from Part 1, which used keyword
search with minsearch. If you skipped Part 1, the next lesson does a
quick revision and walks you through downloading the helpers.

---

# Lesson 12: Quick RAG Revision (Optional)

Source: [`12-rag-revision.md`](../../course/01-agentic-rag/12-rag-revision.md) · Video: [YouTube](https://www.youtube.com/watch?v=gH8fB-6Emmo&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

Before we talk about agents, let's set up the RAG pipeline we built
in Part 1.

Our courses have a lot of participants. They ask the same questions
over and over, so we keep a FAQ document and point students to it. RAG
takes that FAQ and finds the entry that matches a question. It then
sends the entry to an LLM so it can answer. That way a student gets a
reply right away instead of scrolling through a long document.

![Fixed keyword RAG fails on a typo, while an agent rewrites the query, searches again, and reaches an answer](../../course/01-agentic-rag/images/12-rag-revision-01-typo-retry-imagegen.png)


We'll use two helpers we defined earlier in this module:

- [`rag_helper.py`](../../course/01-agentic-rag/code/rag_helper.py) - the `RAGBase` class wrapping search, prompt building, and the LLM call
- [`ingest.py`](../../course/01-agentic-rag/code/ingest.py) - `load_faq_data` and `build_index` for loading the FAQ and building a minsearch index


If you're working through Part 2 as a standalone workshop (without
Part 1), download them into your project:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/01-agentic-rag/code/rag_helper.py
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/01-agentic-rag/code/ingest.py
```

## Setting up RAG

Set up the OpenAI client:

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

Load the data and build the search index:

In [ ]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

Create the assistant:

In [ ]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

## Testing it

Let's try a question:

In [12]:
assistant.rag("How do I run Ollama locally?")

'1. Install Ollama from [ollama.com/download](https://ollama.com/download), choosing the installer for your operating system. On Linux, you can run:\n\n   ```bash\n   curl -fsSL https://ollama.com/install.sh | sh\n   ```\n\n2. Open a terminal and start a model:\n\n   ```bash\n   ollama run llama3\n   ```\n\n   This downloads the LLaMA 3 model (about 4 GB) and opens a local chat.\n\n3. To check that the local server is responding, run:\n\n   ```bash\n   curl http://localhost:11434\n   ```\n\n   You should receive a response containing a `models` field. For Python use, install the client with `pip install ollama`.'

This works fine. The search finds relevant FAQ entries about Ollama,
and the LLM gives a good answer.

Now try something slightly different:

In [ ]:
# note the typo
assistant.rag("How do I run Olama locally?")

"I don't know."

The word "Olama" doesn't match "Ollama" in our index. We use lexical
search, so it looks for the exact word and finds nothing. The LLM
gets these bad results and either says "I don't know" or answers with
irrelevant information.

This is the limitation of a fixed pipeline. The search runs once with
the exact query the user typed, and there's no second chance. The
pipeline doesn't know the search failed, so it can't try again with a
corrected query.

We need something smarter. We need an agent.

---

# Lesson 13: Function Calling

Source: [`13-function-calling.md`](../../course/01-agentic-rag/13-function-calling.md) · Video: [YouTube](https://www.youtube.com/watch?v=CeEki_0mdGo&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson we built a RAG pipeline with `RAGBase.rag()`
and saw it fail on the "Olama" typo. The search returned nothing
useful, and the LLM had no way to recover.

The flow that broke:

```mermaid
flowchart TD
    U([User: How do I run Olama?])
    S[search - Olama - no useful results]
    A([LLM: I don't have information about Olama.])

    U --> S --> A
```

The pipeline is fixed: search, build prompt, LLM.

In [ ]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

The LLM is a passenger here, not a driver. It never sees the bad
search results, so it can't try again with a corrected query.

## The agent alternative

An agent puts the LLM in charge.

Instead of running search ourselves, we give the LLM a `search` tool.
It decides when to call it and what to search for.

The same typo question now goes like this:

```mermaid
flowchart TD
    U([User: How do I run Olama?])
    L1[LLM: I'll search for 'Olama']
    S1[search - Olama - no useful results]
    L2[LLM: Hmm, no results. Maybe a typo for 'Ollama'?]
    S2[search - Ollama - found results!]
    A([LLM: Here's how to run Ollama locally...])

    U --> L1 --> S1 --> L2 --> S2 --> A
```

The LLM searched, saw the results were bad, and decided to try again
with a different query. It made that decision on its own. We didn't
write any code to handle typos.

The difference is about who makes the decisions:

- With RAG, the developer decides. We fix the steps up front, so
  search always runs once with the exact user query.
- With an agent, the LLM decides. It chooses which actions to take
  and when to stop.

The mechanism that makes this possible is function calling, and that's
what the rest of this lesson is about.

## Asking without tools

First, let's see what the LLM does without any tools. We ask it a
course-specific question and look at the answer.

In [ ]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

The model answers from its general knowledge, something like "it
depends on the course" or "check the course website". It doesn't know
about our FAQ, so the answer is vague and not helpful. This is exactly
why we need RAG, and why we want to hand the model a tool.

## Defining the tool

First we define a top-level `search` function that queries the `index`
directly. The model will reference it by this name. We keep the Python
function and the tool name aligned so the dispatch is easier later.

In [14]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Next we tell the model about this function. The model doesn't see our
Python code, only a schema describing what the function does and what
arguments it takes. LLMs are language agnostic. At the end we're just
making an HTTP call, so we describe the tool in JSON rather than in
Python. The same schema would work from TypeScript or Java.

In [15]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

The `description` is the most important field, because the model reads
it to decide when to call the function. `parameters` is a JSON schema
for the arguments, and we mark `query` as required so the model always
fills it in.

## Sending the question with the tool

Now we send the same question as before, but this time we include the
tool in the request:

In [16]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-6-luna",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? late enrollment registration joining course"}', call_id='call_PwtEbveCZajX1kVRKl9xespm', name='search', type='function_call', id='fc_0770b724c1d8b0da006ae0fd7190c487d19041bd4e9ded8bd6', async_=None, caller=None, namespace=None, status='completed')]

Look at the output. Instead of a message with the answer, the response
contains a `function_call` entry. The model decided it needs to search
the FAQ before answering. Rather than reply, it asked us to run the
search function first.

Look at the arguments too. The model didn't pass our question
verbatim. It judged the raw question wasn't the best query to search
with. So it rewrote our enrollment question into search keywords like
"enroll late join course".

## Executing the function and sending the result back

The function call contains JSON arguments. We parse them, call our
`search` function, and serialize the result.

In [17]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

Now we send this result back to the model. First, we add the model's
output to the conversation history - the model needs to see its own
function call. Then we add the tool result.

In [18]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

The `call_id` links the tool result to the specific function call the
model requested. If the model makes multiple function calls in one
turn, each one gets its own `call_id`.

## Asking the model again

We call the API a second time with the expanded history:

In [19]:
response = openai_client.responses.create(
    model="gpt-6-luna",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes! You can join and start learning whenever you like. The materials are available online. If you want a certificate, make sure to submit your project while submissions are still open.'

This time the model has the original question, its own decision to
call `search`, and the FAQ results. It can now produce a proper
course-specific answer.


We have to send the whole history because LLMs are stateless between
API calls. The memory is the list you send as `input`. If you send
only the tool result, the model has no idea what's going on. So on
this second call we replay everything we have so far. That means the
question, the decision to call `search`, and the result we got back.


That's the full function-calling loop for a single turn. With plain
RAG we made one call, and here we make two. Turning RAG agentic means
more round-trips.

People call this pattern "agentic RAG", "tool use", or "function
calling". The idea behind all of them is the same. The LLM decides
which tools to call.

## Token usage and cost

We just made two API calls instead of one. Each call we send to the
model costs money, so it's worth checking how much one tool-using turn
actually costs.

The response has a `usage` field with the token counts:

In [ ]:
usage = response.usage
usage.input_tokens, usage.output_tokens

For each model the provider publishes a price per million input tokens
and per million output tokens. Plug those numbers in to convert tokens
to dollars.

In [ ]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

This usage is only for the second API call. The first call also has
its own usage and its own cost. That was the call where the model
decided to invoke `search`. Two calls means we pay twice. We pay even
more on the second call, because we resend the full history as input.

With a real agent loop the model can make many calls, so the costs add
up. Keep an eye on `usage` while you develop.

---

# Lesson 14: The Agentic Loop

Source: [`14-agentic-loop.md`](../../course/01-agentic-rag/14-agentic-loop.md) · Video: [YouTube](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That works for one function call. It breaks down when the model wants
to search several times, or when the first search misses the answer.
We don't know in advance how many calls the model will want. So we
need a loop that keeps calling the model and running tools until it's
done. An agent is exactly that.

## Anatomy of an agent

With the LLM in the driver's seat, we have an agent. It's an AI
assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the
  `developer` message. The better the instructions, the better the
  agent helps.
- Tools, the functions the agent can call to carry out the task. For
  us that's only `search`.
- Memory, the message history. We append every prompt, every model
  output, and every tool result. The agent reads this to know what it
  has already tried.

Everything below is the code that wires these three together inside a
loop.

## A developer prompt

So far we've relied on the model to figure out when to search. We make
that more reliable with a `developer` message that spells out how to
behave. This is where we give the agent its role. The same message
also pushes it toward multiple searches, so we get to watch the loop
run more than once.

In [20]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

## A function-call helper

We'll be running function calls repeatedly inside the loop, so let's
wrap that in a small helper. It turns the JSON arguments into a Python
dict, calls the right function, and serializes the result. We only
have one tool for now, so we dispatch on the function name directly.

The helper returns the exact structure the Responses API expects.
When we add more tools later, we'll extend this with more `if`
branches (or switch to a registry).

## Processing one response

Let's process a single model response. We append each output entry to
the conversation, print any messages, and run any function calls.
Function-call results get appended too.

In [21]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course late registration can I join discovered the course"}
function_call: search {"query":"new student enroll after course start join course late"}
function_call: search {"query":"course FAQ enrollment late join discovered course"}


The `has_function_calls` flag tells us whether the model needs another
API call. If the response contains a function call, the updated
`messages` has tool output the model hasn't seen yet. We'll need to
send it back.

## The full agent loop

We wrap this in a `while` loop. The loop keeps calling the model until
it returns a response without any function calls. We also keep an
iteration counter so we can see how many round-trips happened.

In [22]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while the course is still accepting submissions. You can also start learning anytime using the course materials.

If you want, I can also help you figure out the best way to start or explain the certificate requirements.


This is the core agent loop. The model reasons about the next action.
Your code performs it, and the model sees the result on the next turn.
The loop stops when the model returns a final answer with no more tool
calls.


We don't decide how many times the model searches. The model does,
and we keep looping until it stops asking for tools.

The exit condition is the simplest one possible. No function calls
this turn means we're done. Other frameworks add safety nets on top,
like a max iteration count, a token budget, or a wall-clock limit. You
might cap it at five iterations and force an answer on the last one.
The core is still this one flag.

## Wrapping it in a function

Let's wrap the loop in a function so we can reuse it. The function
takes the instructions and the question as parameters, and returns
the final answer.

In [23]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

Try it with a question that has a typo:

In [24]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run install setup Ollama locally run model"}
function_call: search {"query":"Ollama local server run model FAQ"}
function_call: search {"query":"run Ollama locally command line ollama serve ollama run"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - macOS: download the `.pkg` from https://ollama.com/download
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model**
   ```bash
   ollama run llama3
   ```
   This will download the model and start a local chat session.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from the Ollama server.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - macOS: download the `.pkg` from https://ollama.com/download\n   - Windows: download the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and start a local chat session.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from the Ollama server.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection error, restarting the server with this can help:\n```bash\nnohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you want, I can also show you how to run Ollama with a 

Watch what happens. The agent searches for "Olama" and gets poor
results. It then searches again with "Ollama" and finds the answer.
The loop lets the model recover from a bad search on its own. That's
the whole point of going agentic.

Also try the course enrollment question:

In [25]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enroll discovered course can I still join"}
function_call: search {"query":"course enrollment late join discovered course FAQ"}
iteration #2...
ASSISTANT:
Yes, you can still join the course.

If you want a certificate, make sure you submit your project while the course is still accepting submissions. You can also go through the materials at your own pace, but certificate-eligible project submission and peer review need to happen during an active/live cohort.

If you'd like, I can also help you figure out the best way to catch up quickly or explain the certificate requirements in more detail. Any other areas you want to explore?


"Yes, you can still join the course.\n\nIf you want a certificate, make sure you submit your project while the course is still accepting submissions. You can also go through the materials at your own pace, but certificate-eligible project submission and peer review need to happen during an active/live cohort.\n\nIf you'd like, I can also help you figure out the best way to catch up quickly or explain the certificate requirements in more detail. Any other areas you want to explore?"

## Encouraging multiple searches

There's a subtle issue here. The model often answers after the first
search, even when more searches would help. It reasons that it already
knows enough, so why bother. We push it to explore more by rewriting
the instructions.

In [26]:
# avoid answering straight away
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late join FAQs"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, you’ll need to submit your project while submissions are still open.

If you want, I can also help explain how the course workflow and certificate rules work.


'Yes — you can still join the course.\n\nIf you want a certificate, you’ll need to submit your project while submissions are still open.\n\nIf you want, I can also help explain how the course workflow and certificate rules work.'

Now the agent makes multiple searches per question and doesn't stop
after the first round of results. The instructions are how we steer
the agent. It can still decide to skip ahead sometimes, so don't
expect it to follow them every single run.

## Restricting off-topic questions

Right now the agent will answer anything. Ask it about chess and it
will still try.

In [ ]:
agent_loop(instructions, "what's queen gambit?")

We want a course assistant, not a general chatbot. We tighten the
instructions so the agent only answers from the FAQ. For our own use
we might be fine letting it answer from general knowledge. So treat
this mainly as an illustration of steering scope.

In [27]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
ASSISTANT:
I couldn’t find a course-related FAQ entry for “queen gambit,” so it looks like this is off-topic for the course.

If you meant something else course-related, feel free to clarify. Are there other areas you want to explore?


'I couldn’t find a course-related FAQ entry for “queen gambit,” so it looks like this is off-topic for the course.\n\nIf you meant something else course-related, feel free to clarify. Are there other areas you want to explore?'

This is a lightweight form of an input guardrail. We tell the agent
what's in scope and what isn't. A real guardrail checks the input
before the agent runs and can block off-topic questions outright.
That's a separate topic, but instructions are the first place to start.

This handwritten loop is the best way to understand what frameworks
hide from you. Every agent framework wraps this same pattern, whether
it's LangChain, PydanticAI, or the OpenAI Agents SDK.

---

# Lesson 15: ToyAIKit

Source: [`15-frameworks.md`](../../course/01-agentic-rag/15-frameworks.md) · Video: [YouTube](https://www.youtube.com/watch?v=PQpQOR3Un3w&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

The handwritten agent loop from the previous lesson is educational but
repetitive. Every time you build a new agent, you'd write the same
while-loop, the same function-call handling, the same message
management.

ToyAIKit wraps this pattern so you can focus on tools, prompts, and
behavior. We built it together in a DataTalks.Club workshop a while
back. It does the same thing as our handwritten loop with less
boilerplate. If you open its `runners` code, you'll find the same
`while True` loop we wrote by hand.



I use it here on purpose, because I don't want to pick a winner among
the production frameworks. ToyAIKit is small and easy to read, so when
something breaks you can see exactly what happened. That makes it handy
for developing and debugging locally before you go to production.

One caveat. ToyAIKit is a teaching and experimentation library, and it
is NOT meant for production use. We use it because it's minimal and you
can see what it does.

## Setup

Install it:

```bash
uv add toyaikit
```

Import the classes we need:

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

## Registering the tool

We register our `search` function along with the schema from earlier
lessons:

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

## Letting ToyAIKit generate the schema

Writing that schema by hand is annoying, and we don't want to do it
for every function. So we don't have to.

If we add a type hint and a docstring to `search`, ToyAIKit reads them
and derives the schema for us:

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

Then register it without passing a schema:

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search)

You can look at what ToyAIKit produced.

In [ ]:
agent_tools.get_tools()

The output is the same JSON schema we hand-wrote in the function
calling lesson. ToyAIKit generated it from the docstring and the type
hint.

Every modern agent framework does this same trick. It reads a typed
Python function with a docstring and builds the schema from it. The
OpenAI Agents SDK, PydanticAI, LangChain and Google ADK all work this
way. You write the tool and the framework figures out how to describe
it.

## The chat interface and runner

Create the chat interface and a callback, then build the runner:

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

The `chat_interface` handles display in the notebook. The `callback`
renders model messages and tool calls as they happen. The runner runs
the agent loop, the same `while True` we wrote by hand. It sends
messages, executes function calls, adds tool outputs back, and repeats
until the model is done.

We pick `gpt-5.4-mini` here on purpose. Without it, ToyAIKit falls
back to a smaller, faster default that doesn't follow the instructions
as reliably.

## Running one prompt

Run a single prompt:

In [ ]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

We used the typo "Olama" on purpose. The agent searches and gets poor
results, then retries with "Ollama". The recovery is the same as the
handwritten loop. The notebook output is nicer to watch. Each tool
call and message renders inline, so you can look at every search
result.


The `result` is a `LoopResult` with `all_messages` (the full
conversation), token counts, and `cost` (computed from token usage).

## Cost and tokens

Look at what the call cost:

In [ ]:
result.cost

Useful while developing - especially with multi-turn agents where one
prompt can trigger several model calls. The handwritten loop made you
compute this by hand. The framework keeps a running total for you.

You can also look at the full message history.

In [ ]:
result.all_messages

This is just a list - the same `messages` list we maintained by hand.

## Continuing the conversation

Take the messages from the previous result and pass them as
`previous_messages` on the next `loop` call:

In [ ]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

The runner picks up where the last call left off, with the same agent
loop and an extended history. The model knows "different model" refers
to Ollama because it sees the previous turn in memory. Without that
history, it would have no idea what we're asking about.

## Interactive chat

For a chat-like workflow, run the built-in input loop:

In [ ]:
runner.run()

Type questions and get answers. Type "stop" to exit.

---

# Lesson 16: Other Frameworks

Source: [`16-other-frameworks.md`](../../course/01-agentic-rag/16-other-frameworks.md) · Video: [YouTube](https://www.youtube.com/watch?v=4yiCbKX9RhI&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

The concepts you learned in Part 2 are the same across frameworks.
Function calling, the agent loop, and tool definitions all wrap the
same pattern. Send messages, run any function calls, and repeat until
the model is done.

![Different agent SDKs share the same core loop of messages, tool calls, observation, and an answer](../../course/01-agentic-rag/images/16-other-frameworks-01-shared-agent-loop-imagegen.png)


You now understand how the loop works. So you can pick up any
production framework and know what it's doing under the hood. I kept
this module framework agnostic on purpose, so you can explore and pick
the one you like.


Here are some frameworks worth a look:

## OpenAI Agents SDK

The official SDK from OpenAI for building agents. It uses the same
Responses API we used throughout this module. It supports tool
definition, multi-turn conversations, and handoffs between agents.

```bash
uv add openai-agents
```

Good choice if you're already using OpenAI and want something
official and well-maintained.

## PydanticAI

A type-safe agent framework that supports multiple LLM providers.
Tools are plain Python functions with type hints, no wrappers needed.
Switching providers is as simple as changing the model string.

```bash
uv add pydantic-ai
```

This is my personal favorite. The reason isn't the type safety, since
the other frameworks have that too. It's how it feels to use, and the
team behind Pydantic. Good choice if you want type safety and
multi-provider support.

## LangChain / LangGraph

A popular framework with lots of integrations. LangChain handles the
basics, and LangGraph adds graph-based workflows for more complex
agent patterns.

Good choice if you need lots of integrations (vector stores, document
loaders, etc.) and a large community.

## Google ADK

The Agent Development Kit from Google. If you plan to use Gemini
models, this is the one I'd reach for. It exposes the same building
blocks we've seen, like tools, instructions, and sessions. It also
integrates with Google Cloud.

Good choice if your stack is on Google Cloud or you specifically want
Gemini.

## Others

Other frameworks worth knowing:

- CrewAI - multi-agent orchestration
- AutoGen - multi-agent conversations from Microsoft
- Semantic Kernel - from Microsoft, supports C# and Python
- Smolagents - lightweight agent framework from HuggingFace
- Anthropic Tool Use - Anthropic's native tool use API

Pick one that fits your stack and your needs. The hard part is
designing good tools and prompts - the loop is always the same.

## Avoiding agents when you can

One closing thought. You don't always need an agent, and that's fine.
You might assume your app needs one, but often it doesn't.

Adding an agent is a real cost:

- More API calls per request, since the loop can fire many tool calls
  before the model is satisfied.
- Higher latency, since each round-trip waits on the model.
- More money spent, since every iteration is another billed call and
  the full message history is re-sent each turn.
- More moving parts, since you now monitor cost, iteration count, and
  whether the agent is going in circles.
- Less predictable behavior, since the LLM decides what to do and two
  runs of the same prompt can take different paths.


Before reaching for an agent, ask whether the problem needs one. A lot
of tasks are well served by simpler approaches.

Reach for one of these first:

- plain RAG, with one search and one answer
- parsing or templating a document into another form
- a single LLM call with no tools

Try the simpler approach first, and if it works, ship it. Reach for
the agent loop only when you've tried the simpler solution and it
genuinely can't handle the problem. By then you'll know the extra
complexity is worth it, and you'll be ready to take it on.